# 11. PyTorch Tutorial 11 - Softmax and Cross Entropy

In [20]:
import torch
import torch.nn as nn
import numpy as np

### Custom softmax

In [21]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=0)

x = np.array([2.0, 1.0, 0.1])
outputs = softmax(x)
print('softmax numpy:', outputs)

softmax numpy: [0.65900114 0.24243297 0.09856589]


### torch softmax

In [22]:
x = torch.tensor([2.0, 1.0, 0.1])
outputs = torch.softmax(x, dim=0) # along values along first axis
print('softmax torch:', outputs)

softmax torch: tensor([0.6590, 0.2424, 0.0986])


### Custom Cross Entropy

`np.clip()` is a function in the NumPy library of Python that is used to limit the elements in an array. The function takes an interval (combination of minimum value and maximum value), and values outside the interval are clipped to the interval edges. For example, if an interval of [0, 1] is specified, values smaller than 0 become 0, and values larger than 1 become 1.

In [23]:
def cross_entropy(actual, predicted):
    EPS = 1e-15 # EPS stands for Epsilon (it helps avoid zero division errors of log errors)
    predicted = np.clip(predicted, EPS, 1 - EPS)
    loss = -np.sum(actual * np.log(predicted)) # element-wise multiplication
    return loss # / float(predicted.shape[0])

In [24]:
np.log(np.array([0.7, 0.2, 0.1])) * np.array([1, 0, 0])

array([-0.35667494, -0.        , -0.        ])

In [25]:
Y = np.array([1, 0, 0])
Y_pred_good = np.array([0.7, 0.2, 0.1])
Y_pred_bad = np.array([0.1, 0.3, 0.6])
l1 = cross_entropy(Y, Y_pred_good)
l2 = cross_entropy(Y, Y_pred_bad)
print(f'Loss1 numpy: {l1:.4f}')
print(f'Loss2 numpy: {l2:.4f}')

Loss1 numpy: 0.3567
Loss2 numpy: 2.3026


### torch Cross Entropy : nn.CrossEntropyLoss()

In [26]:
loss = nn.CrossEntropyLoss()
# loss(input, target)

# target is of size nSamples = 1
# each element has class label: 0, 1, or 2
# Y (=target) contains class labels, not one-hot
Y_0 = torch.tensor([0])

In [27]:
# input is of size nSamples x nClasses = 1 x 3
# y_pred (=input) must be raw, unnormalizes scores (logits) for each class, not softmax
Y_pred_good = torch.tensor([[2.0, 1.0, 0.1]])
Y_pred_bad = torch.tensor([[0.5, 2.0, 0.3]])
l1 = loss(Y_pred_good, Y_0)
l2 = loss(Y_pred_bad, Y_0)

print(f'PyTorch Loss1 good: {l1.item():.4f}')
print(f'PyTorch Loss2 bad: {l2.item():.4f}')

PyTorch Loss1 good: 0.4170
PyTorch Loss2 bad: 1.8406


In [28]:
Y_1 = torch.tensor([1])
Y_pred_bad = torch.tensor([[2.0, 1.0, 0.1]])
Y_pred_good = torch.tensor([[0.5, 2.0, 0.3]])
l1 = loss(Y_pred_bad, Y_1)
l2 = loss(Y_pred_good, Y_1)

print(f'PyTorch Loss1 bad: {l1.item():.4f}')
print(f'PyTorch Loss2 good: {l2.item():.4f}')

PyTorch Loss1 bad: 1.4170
PyTorch Loss2 good: 0.3406


### get predictions

In [29]:
_, predictions1 = torch.max(Y_pred_good, 1)
_, predictions2 = torch.max(Y_pred_bad, 1)
print(f'Actual class: {Y.item()}, Y_pred1: {predictions1.item()}, Y_pred2: {predictions2.item()}')

ValueError: can only convert an array of size 1 to a Python scalar

In [33]:
torch.max?

In [ ]:
# allows batch loss for multiple samples

# target is of size nBatch = 3
# each element has class label: 0, 1, or 2
Y = torch.tensor([2, 0, 1])

# input is of size nBatch x nClasses = 3 x 3
# Y_pred are logits (not softmax)
Y_pred_good = torch.tensor(
    [[0.1, 0.2, 3.9], # predict class 2
    [1.2, 0.1, 0.3], # predict class 0
    [0.3, 2.2, 0.2]]) # predict class 1

Y_pred_bad = torch.tensor(
    [[0.9, 0.2, 0.1],
    [0.1, 0.3, 1.5],
    [1.2, 0.2, 0.5]])

l1 = loss(Y_pred_good, Y)
l2 = loss(Y_pred_bad, Y)
print(f'Batch Loss1:  {l1.item():.4f}')
print(f'Batch Loss2: {l2.item():.4f}')

# get predictions
_, predictions1 = torch.max(Y_pred_good, 1)
_, predictions2 = torch.max(Y_pred_bad, 1)
print(f'Actual class: {Y}, Y_pred1: {predictions1}, Y_pred2: {predictions2}')

# Binary classification
class NeuralNet1(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(NeuralNet1, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, 1)  
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # sigmoid at the end
        y_pred = torch.sigmoid(out)
        return y_pred

model = NeuralNet1(input_size=28*28, hidden_size=5)
criterion = nn.BCELoss()

# Multiclass problem
class NeuralNet2(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet2, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, num_classes)  
    
    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        # no softmax at the end
        return out

model = NeuralNet2(input_size=28*28, hidden_size=5, num_classes=3)
criterion = nn.CrossEntropyLoss()  # (applies Softmax)